# Chapter 12.3: Serving Long-Context Models -- 128K, 1M+ Context

## Overview

Long-context LLMs (128K--1M+ tokens) present unique serving challenges. The quadratic attention complexity, massive KV-cache requirements, and high memory bandwidth demands require specialized techniques.

### Key Challenges
1. **Memory**: KV-cache for 1M tokens on a 70B model requires ~1.3 TB
2. **Compute**: Self-attention is O(n^2) -- 1M tokens is ~1000x more expensive than 1K
3. **Latency**: Prefill of 128K tokens takes seconds on a single GPU

### Solutions
- Ring Attention & Sequence Parallelism
- KV-cache offloading to CPU/SSD
- Chunked prefill strategies
- Sparse/linear attention approximations

### Learning Objectives
1. Calculate memory and compute requirements for long context
2. Implement ring attention simulation
3. Profile attention computation time vs context length
4. Design offloading strategies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part12/chapter_12.3_long_context.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part12/chapter_12.3_long_context.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

np.random.seed(42)
plt.style.use('default')
print("Long-context serving environment ready.")

## 1. Memory and Compute Requirements Analysis

In [ ]:
@dataclass
class LongContextModel:
    name: str
    num_layers: int
    hidden_dim: int
    num_kv_heads: int  # GQA heads
    head_dim: int
    max_context: int
    param_bytes: int = 2  # FP16

    @property
    def kv_per_token_bytes(self):
        return 2 * self.num_layers * self.num_kv_heads * self.head_dim * self.param_bytes

    def kv_cache_gb(self, seq_len: int):
        return self.kv_per_token_bytes * seq_len / (1024**3)

    def attention_flops(self, seq_len: int):
        """FLOPs for self-attention: 2 * n^2 * d per layer per head."""
        return self.num_layers * self.num_kv_heads * 2 * seq_len * seq_len * self.head_dim

    @property
    def model_size_gb(self):
        params = 12 * self.num_layers * self.hidden_dim ** 2
        return params * self.param_bytes / (1024**3)


models = {
    'LLaMA-3.1-8B': LongContextModel('LLaMA-3.1-8B', 32, 4096, 8, 128, 131072),
    'LLaMA-3.1-70B': LongContextModel('LLaMA-3.1-70B', 80, 8192, 8, 128, 131072),
    'Gemini-1M': LongContextModel('Gemini-1M-est', 80, 8192, 8, 128, 1048576),
    'Claude-200K': LongContextModel('Claude-200K-est', 80, 8192, 8, 128, 200000),
}

context_lengths = [1024, 4096, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]

print(f"{'Model':>20} {'Params (GB)':>12} {'KV/token (B)':>12} {'Max Ctx':>10}")
print("-" * 60)
for name, m in models.items():
    print(f"{name:>20} {m.model_size_gb:>10.1f}GB {m.kv_per_token_bytes:>10.0f}B {m.max_context:>10,}")

print(f"\nKV-Cache Memory Requirements (GB):")
print(f"{'Context':>10}", end='')
for name in models:
    print(f"{name:>20}", end='')
print()
print("-" * (10 + 20 * len(models)))
for cl in context_lengths:
    print(f"{cl:>10,}", end='')
    for name, m in models.items():
        kv_gb = m.kv_cache_gb(cl)
        print(f"{kv_gb:>18.1f}GB", end='')
    print()

In [ ]:
# Compute and memory scaling visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ctx = np.array([1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576])
model = models['LLaMA-3.1-70B']

# KV-cache memory
kv_mem = [model.kv_cache_gb(c) for c in ctx]
axes[0].loglog(ctx, kv_mem, 'bo-', linewidth=2)
axes[0].axhline(80, color='red', linestyle='--', label='80GB GPU')
axes[0].axhline(80*8, color='green', linestyle='--', label='8x 80GB GPUs')
axes[0].set_xlabel('Context Length')
axes[0].set_ylabel('KV-Cache Memory (GB)')
axes[0].set_title('KV-Cache Memory Scaling')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Attention FLOPs
attn_flops = [model.attention_flops(c) / 1e12 for c in ctx]  # TFLOPS
axes[1].loglog(ctx, attn_flops, 'ro-', linewidth=2)
axes[1].set_xlabel('Context Length')
axes[1].set_ylabel('Attention FLOPs (TFLOPS)')
axes[1].set_title('Attention Compute Scaling (O(n^2))')
axes[1].grid(True, alpha=0.3)

# Prefill time estimate on H100
h100_tflops = 989
prefill_times = []
for c in ctx:
    attn = model.attention_flops(c)
    total_params = 12 * model.num_layers * model.hidden_dim ** 2
    ffn = 2 * total_params * c  # FFN + MLP FLOPs
    total = (attn + ffn) / (h100_tflops * 1e12)
    prefill_times.append(total)

axes[2].loglog(ctx, prefill_times, 'go-', linewidth=2, label='Single H100')
axes[2].loglog(ctx, [t/8 for t in prefill_times], 'g--', linewidth=1, label='8x H100 (ideal)')
axes[2].axhline(1, color='orange', linestyle='--', alpha=0.5, label='1 second')
axes[2].axhline(10, color='red', linestyle='--', alpha=0.5, label='10 seconds')
axes[2].set_xlabel('Context Length')
axes[2].set_ylabel('Prefill Time (seconds)')
axes[2].set_title('Estimated Prefill Time')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle('LLaMA-3.1-70B Long Context Scaling', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('long_context_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Demo: Ring Attention Simulation

Ring Attention distributes long sequences across multiple devices in a ring topology. Each device computes attention for its local chunk while passing KV-cache blocks around the ring.

**Algorithm:**
1. Split sequence into N chunks (one per device)
2. Each device holds query chunk locally
3. KV blocks rotate around the ring
4. Each step: compute partial attention with current KV block, pass KV to next device
5. After N steps, each device has attended to all tokens

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np

def draw_ring_attention():
    """Ring attention diagram: sequence split across GPUs in ring topology."""
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 14); ax.set_ylim(0, 14); ax.axis('off')
    ax.set_title('Ring Attention: Distributed Long-Context Processing',
                 fontsize=15, fontweight='bold', pad=15)

    # Draw GPUs in a ring
    n_gpus = 4
    center_x, center_y = 7, 7
    radius = 4.0
    gpu_colors = ['#4A90D9', '#27AE60', '#F39C12', '#E74C3C']
    chunk_labels = ['Chunk 0\n(tok 0-1K)', 'Chunk 1\n(tok 1K-2K)', 'Chunk 2\n(tok 2K-3K)', 'Chunk 3\n(tok 3K-4K)']

    angles = np.linspace(np.pi/2, np.pi/2 + 2*np.pi, n_gpus, endpoint=False)
    gpu_positions = []

    for i in range(n_gpus):
        x = center_x + radius * np.cos(angles[i])
        y = center_y + radius * np.sin(angles[i])
        gpu_positions.append((x, y))

        # GPU box
        gpu = FancyBboxPatch((x - 1.5, y - 1.0), 3.0, 2.0, boxstyle='round,pad=0.12',
                             facecolor=gpu_colors[i], edgecolor='white', linewidth=2, alpha=0.85)
        ax.add_patch(gpu)
        ax.text(x, y + 0.35, f'GPU {i}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='white')
        ax.text(x, y - 0.35, chunk_labels[i], ha='center', va='center',
                fontsize=7, color='white', alpha=0.9)

    # Draw ring arrows (KV passing)
    for i in range(n_gpus):
        next_i = (i + 1) % n_gpus
        x1, y1 = gpu_positions[i]
        x2, y2 = gpu_positions[next_i]

        # Calculate midpoint slightly inward for the arrow
        mid_x = (x1 + x2) / 2
        mid_y = (y1 + y2) / 2

        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color='#8E44AD', lw=2.5,
                                    connectionstyle='arc3,rad=0.3'))

    # Center label
    ax.text(center_x, center_y + 0.5, 'KV blocks\nrotate around\nthe ring', ha='center', va='center',
            fontsize=11, fontweight='bold', color='#8E44AD',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#F3E5F5', edgecolor='#8E44AD'))

    ax.text(center_x, center_y - 1.0, 'Each GPU:\n1. Compute attention with local KV\n2. Pass KV to next GPU\n3. Repeat N times',
            ha='center', va='center', fontsize=8, color='#555')

    # Sequence diagram at bottom
    ax.text(7, 0.8, 'Full Sequence (4K tokens)', ha='center', fontsize=10, fontweight='bold', color='#333')
    for i in range(4):
        x = 2.5 + i * 2.3
        rect = FancyBboxPatch((x, 0.0), 2.0, 0.6, boxstyle='round,pad=0.05',
                              facecolor=gpu_colors[i], edgecolor='white', linewidth=1.5, alpha=0.7)
        ax.add_patch(rect)
        ax.text(x + 1.0, 0.3, f'Chunk {i}', ha='center', va='center',
                fontsize=8, fontweight='bold', color='white')

    plt.tight_layout()
    plt.savefig('ring_attention.png', dpi=150, bbox_inches='tight')
    plt.show()

draw_ring_attention()

In [ ]:
class RingAttentionSimulator:
    """Simulate Ring Attention across multiple devices."""

    def __init__(self, num_devices: int, seq_len: int, head_dim: int,
                 compute_tflops: float = 989, bandwidth_gbps: float = 450):
        self.num_devices = num_devices
        self.seq_len = seq_len
        self.head_dim = head_dim
        self.chunk_size = seq_len // num_devices
        self.compute_tflops = compute_tflops
        self.bandwidth_gbps = bandwidth_gbps

    def standard_attention(self, queries, keys, values):
        """Standard self-attention (for reference)."""
        scale = 1.0 / np.sqrt(self.head_dim)
        scores = np.matmul(queries, keys.T) * scale
        # Causal mask
        mask = np.triu(np.ones_like(scores) * -1e9, k=1)
        scores = scores + mask
        attn = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attn = attn / np.sum(attn, axis=-1, keepdims=True)
        return np.matmul(attn, values)

    def ring_attention(self, queries, keys, values):
        """Ring attention with online softmax."""
        n = self.seq_len
        cs = self.chunk_size
        nd = self.num_devices
        scale = 1.0 / np.sqrt(self.head_dim)

        # Split into chunks
        q_chunks = [queries[i*cs:(i+1)*cs] for i in range(nd)]
        k_chunks = [keys[i*cs:(i+1)*cs] for i in range(nd)]
        v_chunks = [values[i*cs:(i+1)*cs] for i in range(nd)]

        # Initialize accumulators for each device
        outputs = [np.zeros((cs, self.head_dim)) for _ in range(nd)]
        max_scores = [np.full((cs, 1), -np.inf) for _ in range(nd)]
        sum_exp = [np.zeros((cs, 1)) for _ in range(nd)]

        # Ring rotation: N steps
        for step in range(nd):
            for device in range(nd):
                # Which KV chunk this device currently has
                kv_owner = (device + step) % nd
                q = q_chunks[device]
                k = k_chunks[kv_owner]
                v = v_chunks[kv_owner]

                # Compute local attention scores
                local_scores = np.matmul(q, k.T) * scale  # [cs, cs]

                # Apply causal mask
                q_start = device * cs
                k_start = kv_owner * cs
                for qi in range(cs):
                    for ki in range(cs):
                        if q_start + qi < k_start + ki:
                            local_scores[qi, ki] = -1e9

                # Online softmax update (FlashAttention-style)
                local_max = np.max(local_scores, axis=-1, keepdims=True)
                new_max = np.maximum(max_scores[device], local_max)

                # Rescale existing accumulator
                old_scale_factor = np.exp(max_scores[device] - new_max)
                new_scale_factor = np.exp(local_max - new_max)

                local_exp = np.exp(local_scores - local_max)
                local_sum = np.sum(local_exp, axis=-1, keepdims=True)

                # Update accumulators
                outputs[device] = (outputs[device] * sum_exp[device] * old_scale_factor +
                                    np.matmul(local_exp, v) * new_scale_factor)
                sum_exp[device] = sum_exp[device] * old_scale_factor + local_sum * new_scale_factor
                max_scores[device] = new_max

        # Normalize
        for device in range(nd):
            outputs[device] = outputs[device] / (sum_exp[device] + 1e-10)

        return np.concatenate(outputs, axis=0)

    def estimate_time(self):
        """Estimate ring attention execution time."""
        cs = self.chunk_size
        nd = self.num_devices

        # Compute time: each device does cs x cs attention, nd times
        flops_per_step = 2 * cs * cs * self.head_dim  # Q*K^T + attn*V
        compute_time_per_step = flops_per_step / (self.compute_tflops * 1e12)

        # Communication: send KV chunk to next device each step
        kv_bytes = 2 * cs * self.head_dim * 2  # K and V, FP16
        comm_time_per_step = kv_bytes / (self.bandwidth_gbps * 1e9)

        # Overlapped: total time = max(compute, comm) * nd steps
        step_time = max(compute_time_per_step, comm_time_per_step)
        total_time = step_time * nd

        # Standard attention time (single device)
        standard_flops = 2 * self.seq_len * self.seq_len * self.head_dim
        standard_time = standard_flops / (self.compute_tflops * 1e12)

        return {
            'ring_time': total_time,
            'standard_time': standard_time,
            'speedup': standard_time / total_time,
            'compute_per_step': compute_time_per_step,
            'comm_per_step': comm_time_per_step,
            'bottleneck': 'compute' if compute_time_per_step > comm_time_per_step else 'comm',
        }


# Verify correctness with small example
seq_len, head_dim = 64, 32
sim = RingAttentionSimulator(num_devices=4, seq_len=seq_len, head_dim=head_dim)

Q = np.random.randn(seq_len, head_dim).astype(np.float32) * 0.1
K = np.random.randn(seq_len, head_dim).astype(np.float32) * 0.1
V = np.random.randn(seq_len, head_dim).astype(np.float32) * 0.1

standard_out = sim.standard_attention(Q, K, V)
ring_out = sim.ring_attention(Q, K, V)

mse = np.mean((standard_out - ring_out) ** 2)
max_err = np.max(np.abs(standard_out - ring_out))
print(f"Ring Attention Correctness Check:")
print(f"  MSE: {mse:.2e}")
print(f"  Max absolute error: {max_err:.2e}")
print(f"  Match: {'YES' if max_err < 1e-4 else 'NO (check implementation)'}")

In [ ]:
# Performance analysis of ring attention
device_counts = [2, 4, 8, 16, 32, 64]
context_lengths = [8192, 32768, 131072, 524288, 1048576]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Speedup vs device count for different context lengths
for cl in context_lengths:
    speedups = []
    for nd in device_counts:
        if cl // nd >= 64:  # Minimum chunk size
            sim = RingAttentionSimulator(nd, cl, 128)
            result = sim.estimate_time()
            speedups.append(result['speedup'])
        else:
            speedups.append(None)
    valid = [(d, s) for d, s in zip(device_counts, speedups) if s is not None]
    if valid:
        ds, ss = zip(*valid)
        axes[0].plot(ds, ss, 'o-', label=f'{cl//1024}K')

axes[0].plot(device_counts, device_counts, 'k--', alpha=0.3, label='Linear')
axes[0].set_xlabel('Number of Devices')
axes[0].set_ylabel('Speedup')
axes[0].set_title('Ring Attention Speedup')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Compute vs communication bottleneck
for nd in [4, 8, 16]:
    ctx_range = np.logspace(np.log10(4096), np.log10(1048576), 20).astype(int)
    ctx_range = ctx_range - ctx_range % nd  # Make divisible
    ratios = []
    for cl in ctx_range:
        sim = RingAttentionSimulator(nd, cl, 128)
        result = sim.estimate_time()
        ratios.append(result['compute_per_step'] / result['comm_per_step'])
    axes[1].semilogx(ctx_range, ratios, label=f'{nd} devices')

axes[1].axhline(1, color='red', linestyle='--', alpha=0.5, label='Balanced')
axes[1].set_xlabel('Context Length')
axes[1].set_ylabel('Compute/Comm Ratio')
axes[1].set_title('Compute vs Communication Bottleneck')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Memory per device
model = models['LLaMA-3.1-70B']
for nd in device_counts:
    mems = [model.kv_cache_gb(cl) / nd for cl in context_lengths]
    axes[2].semilogy(context_lengths, mems, 'o-', label=f'{nd} devices')

axes[2].axhline(80, color='red', linestyle='--', alpha=0.5, label='80GB limit')
axes[2].set_xlabel('Context Length')
axes[2].set_ylabel('KV-Cache per Device (GB)')
axes[2].set_title('Memory per Device (70B model)')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)
axes[2].set_xscale('log')

plt.tight_layout()
plt.savefig('ring_attention_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demo: Attention Computation Time Profiling

In [ ]:
def profile_attention_time(context_lengths: List[int], head_dim: int = 128,
                            num_trials: int = 3) -> dict:
    """Profile actual attention computation time for various context lengths."""
    results = []
    for cl in context_lengths:
        if cl > 8192:  # Skip very long for actual computation
            # Estimate based on O(n^2) scaling
            base_time = results[-1]['time'] if results else 0.001
            base_cl = results[-1]['context_length'] if results else 1024
            estimated_time = base_time * (cl / base_cl) ** 2
            results.append({
                'context_length': cl,
                'time': estimated_time,
                'estimated': True
            })
            continue

        times = []
        for _ in range(num_trials):
            Q = np.random.randn(cl, head_dim).astype(np.float32) * 0.1
            K = np.random.randn(cl, head_dim).astype(np.float32) * 0.1
            V = np.random.randn(cl, head_dim).astype(np.float32) * 0.1

            start = time.perf_counter()
            scores = Q @ K.T / np.sqrt(head_dim)
            attn = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
            attn = attn / np.sum(attn, axis=-1, keepdims=True)
            output = attn @ V
            end = time.perf_counter()
            times.append(end - start)

        results.append({
            'context_length': cl,
            'time': np.median(times),
            'estimated': False
        })

    return results

# Profile
profile_ctx = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]
timing_results = profile_attention_time(profile_ctx)

print(f"{'Context':>10} {'Time (s)':>12} {'Type':>10} {'vs 1K ratio':>12}")
print("-" * 50)
base_time = None
for r in timing_results:
    if r['context_length'] == 1024:
        base_time = r['time']
    ratio = r['time'] / base_time if base_time else 1
    print(f"{r['context_length']:>10,} {r['time']:>12.4f} "
          f"{'estimated' if r['estimated'] else 'measured':>10} {ratio:>10.1f}x")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ctx_vals = [r['context_length'] for r in timing_results]
time_vals = [r['time'] for r in timing_results]
measured = [not r['estimated'] for r in timing_results]

ax1.loglog(ctx_vals, time_vals, 'bo-', linewidth=2)
# Theoretical O(n^2) line
ref_idx = 2  # 1024
theoretical = [time_vals[ref_idx] * (c / ctx_vals[ref_idx])**2 for c in ctx_vals]
ax1.loglog(ctx_vals, theoretical, 'r--', alpha=0.5, label='O(n^2) theoretical')
ax1.set_xlabel('Context Length')
ax1.set_ylabel('Time (s)')
ax1.set_title('Attention Time vs Context Length')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Memory footprint of attention matrix
attn_memory = [(c * c * 4) / 1024**3 for c in ctx_vals]  # FP32 attention matrix
ax2.loglog(ctx_vals, attn_memory, 'ro-', linewidth=2)
ax2.axhline(80, color='gray', linestyle='--', label='80GB GPU')
ax2.set_xlabel('Context Length')
ax2.set_ylabel('Attention Matrix Size (GB)')
ax2.set_title('Attention Matrix Memory (per head)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('attention_profiling.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Demo: KV-Cache Offloading to CPU/SSD

For very long contexts, we can offload portions of KV-cache to CPU memory or SSD, loading them back on demand.

In [ ]:
class KVCacheOffloader:
    """Simulate KV-cache offloading to CPU/SSD with prefetching."""

    def __init__(self, gpu_budget_tokens: int, cpu_budget_tokens: int,
                 ssd_budget_tokens: int,
                 gpu_to_cpu_bw: float = 64.0,   # GB/s (PCIe 5.0)
                 cpu_to_ssd_bw: float = 7.0,     # GB/s (NVMe)
                 kv_per_token_bytes: int = 2 * 80 * 8 * 128 * 2):  # 70B model
        self.gpu_budget = gpu_budget_tokens
        self.cpu_budget = cpu_budget_tokens
        self.ssd_budget = ssd_budget_tokens
        self.gpu_to_cpu_bw = gpu_to_cpu_bw
        self.cpu_to_ssd_bw = cpu_to_ssd_bw
        self.kv_per_token = kv_per_token_bytes

        # State tracking
        self.gpu_tokens = []  # Token indices on GPU
        self.cpu_tokens = []  # Token indices on CPU
        self.ssd_tokens = []  # Token indices on SSD

        # Statistics
        self.gpu_hits = 0
        self.cpu_hits = 0
        self.ssd_hits = 0
        self.total_transfer_time = 0

    def add_token(self, token_idx: int):
        """Add a new token to the cache hierarchy."""
        self.gpu_tokens.append(token_idx)

        # Evict from GPU to CPU if over budget
        while len(self.gpu_tokens) > self.gpu_budget:
            evicted = self.gpu_tokens.pop(0)  # FIFO eviction
            self.cpu_tokens.append(evicted)
            self.total_transfer_time += self.kv_per_token / (self.gpu_to_cpu_bw * 1e9)

        # Evict from CPU to SSD if over budget
        while len(self.cpu_tokens) > self.cpu_budget:
            evicted = self.cpu_tokens.pop(0)
            self.ssd_tokens.append(evicted)
            self.total_transfer_time += self.kv_per_token / (self.cpu_to_ssd_bw * 1e9)

    def access_token(self, token_idx: int) -> float:
        """Access a token's KV-cache. Returns access time."""
        if token_idx in self.gpu_tokens:
            self.gpu_hits += 1
            return 0  # Already on GPU

        if token_idx in self.cpu_tokens:
            self.cpu_hits += 1
            # Load from CPU to GPU
            load_time = self.kv_per_token / (self.gpu_to_cpu_bw * 1e9)
            self.total_transfer_time += load_time
            return load_time

        if token_idx in self.ssd_tokens:
            self.ssd_hits += 1
            # Load from SSD to CPU to GPU
            load_time = (self.kv_per_token / (self.cpu_to_ssd_bw * 1e9) +
                        self.kv_per_token / (self.gpu_to_cpu_bw * 1e9))
            self.total_transfer_time += load_time
            return load_time

        return 0  # Token not in cache (shouldn't happen)

    def stats(self):
        total = self.gpu_hits + self.cpu_hits + self.ssd_hits
        return {
            'gpu_tokens': len(self.gpu_tokens),
            'cpu_tokens': len(self.cpu_tokens),
            'ssd_tokens': len(self.ssd_tokens),
            'gpu_hit_rate': self.gpu_hits / total if total > 0 else 0,
            'cpu_hit_rate': self.cpu_hits / total if total > 0 else 0,
            'ssd_hit_rate': self.ssd_hits / total if total > 0 else 0,
            'total_transfer_time_ms': self.total_transfer_time * 1000,
        }


# Simulate long-context serving with offloading
context_length = 131072  # 128K tokens

# GPU can hold 16K tokens, CPU 64K, rest on SSD
offloader = KVCacheOffloader(
    gpu_budget_tokens=16384,
    cpu_budget_tokens=65536,
    ssd_budget_tokens=context_length,
)

# Simulate prefill: add all tokens
for i in range(context_length):
    offloader.add_token(i)

# Simulate decode: access patterns (recent + some long-range)
access_pattern = []
for step in range(1000):
    # 70% recent tokens, 20% medium-range, 10% long-range
    r = np.random.random()
    if r < 0.7:
        token = np.random.randint(context_length - 1024, context_length)
    elif r < 0.9:
        token = np.random.randint(context_length - 16384, context_length)
    else:
        token = np.random.randint(0, context_length)
    access_pattern.append(token)
    offloader.access_token(token)

stats = offloader.stats()
print(f"KV-Cache Offloading Statistics (128K context):")
print(f"  GPU: {stats['gpu_tokens']:,} tokens ({stats['gpu_hit_rate']:.1%} hit rate)")
print(f"  CPU: {stats['cpu_tokens']:,} tokens ({stats['cpu_hit_rate']:.1%} hit rate)")
print(f"  SSD: {stats['ssd_tokens']:,} tokens ({stats['ssd_hit_rate']:.1%} hit rate)")
print(f"  Total transfer overhead: {stats['total_transfer_time_ms']:.1f} ms")

In [ ]:
# Compare different offloading configurations
configs = [
    {'name': 'All GPU (80GB)', 'gpu': 131072, 'cpu': 0, 'ssd': 0},
    {'name': 'GPU 16K + CPU', 'gpu': 16384, 'cpu': 131072, 'ssd': 0},
    {'name': 'GPU 16K + CPU 64K + SSD', 'gpu': 16384, 'cpu': 65536, 'ssd': 131072},
    {'name': 'GPU 32K + CPU 64K + SSD', 'gpu': 32768, 'cpu': 65536, 'ssd': 131072},
    {'name': 'GPU 4K + SSD only', 'gpu': 4096, 'cpu': 0, 'ssd': 131072},
]

comparison = []
for cfg in configs:
    off = KVCacheOffloader(
        gpu_budget_tokens=cfg['gpu'],
        cpu_budget_tokens=cfg['cpu'],
        ssd_budget_tokens=max(cfg['ssd'], 131072),
    )
    for i in range(context_length):
        off.add_token(i)
    for token in access_pattern:
        off.access_token(token)
    s = off.stats()
    s['name'] = cfg['name']
    comparison.append(s)

print(f"{'Config':>30} {'GPU Hit':>10} {'CPU Hit':>10} {'SSD Hit':>10} {'Transfer (ms)':>14}")
print("-" * 80)
for c in comparison:
    print(f"{c['name']:>30} {c['gpu_hit_rate']:>9.1%} {c['cpu_hit_rate']:>9.1%} "
          f"{c['ssd_hit_rate']:>9.1%} {c['total_transfer_time_ms']:>12.1f}ms")

## 5. Chunked Prefill Strategies

For very long prompts, we can split prefill into chunks to manage memory and enable pipelining.

In [ ]:
class ChunkedPrefill:
    """Simulate chunked prefill for long prompts."""

    def __init__(self, model: LongContextModel, gpu_memory_gb: float = 80,
                 compute_tflops: float = 989):
        self.model = model
        self.gpu_memory = gpu_memory_gb
        self.compute = compute_tflops

    def estimate_chunk_time(self, chunk_size: int, context_so_far: int) -> float:
        """Time to process one chunk (attention to all prior context + self)."""
        total_context = context_so_far + chunk_size
        # Attention FLOPs: chunk attends to all prior + self
        attn_flops = 2 * chunk_size * total_context * self.model.head_dim * self.model.num_kv_heads * self.model.num_layers
        # FFN FLOPs
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        ffn_flops = 2 * total_params * chunk_size
        return (attn_flops + ffn_flops) / (self.compute * 1e12)

    def simulate_chunked_prefill(self, total_len: int, chunk_size: int) -> dict:
        """Simulate processing a long prompt in chunks."""
        num_chunks = (total_len + chunk_size - 1) // chunk_size
        chunk_times = []
        context_so_far = 0

        for i in range(num_chunks):
            current_chunk = min(chunk_size, total_len - context_so_far)
            t = self.estimate_chunk_time(current_chunk, context_so_far)
            chunk_times.append(t)
            context_so_far += current_chunk

        return {
            'num_chunks': num_chunks,
            'chunk_times': chunk_times,
            'total_time': sum(chunk_times),
            'kv_memory_gb': self.model.kv_cache_gb(total_len),
            'peak_activation_gb': chunk_size * self.model.hidden_dim * 4 / 1024**3,  # FP32 activations
        }


cp = ChunkedPrefill(models['LLaMA-3.1-70B'])

total_len = 131072
chunk_sizes = [1024, 2048, 4096, 8192, 16384, 32768, 65536]

print(f"Chunked Prefill Analysis (131K tokens):")
print(f"{'Chunk Size':>12} {'Chunks':>8} {'Total Time':>12} {'Peak Activation':>16} {'KV Memory':>12}")
print("-" * 65)
for cs in chunk_sizes:
    result = cp.simulate_chunked_prefill(total_len, cs)
    print(f"{cs:>12,} {result['num_chunks']:>8} {result['total_time']:>10.2f}s "
          f"{result['peak_activation_gb']:>14.2f}GB {result['kv_memory_gb']:>10.1f}GB")

# Visualize chunk time progression
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for cs in [2048, 8192, 32768]:
    result = cp.simulate_chunked_prefill(total_len, cs)
    times = result['chunk_times']
    cumulative = np.cumsum(times)
    ax1.plot(range(len(times)), [t * 1000 for t in times], label=f'Chunk={cs}')
    ax2.plot(range(len(cumulative)), cumulative, label=f'Chunk={cs}')

ax1.set_xlabel('Chunk Index')
ax1.set_ylabel('Chunk Time (ms)')
ax1.set_title('Per-Chunk Processing Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Chunk Index')
ax2.set_ylabel('Cumulative Time (s)')
ax2.set_title('Cumulative Prefill Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('chunked_prefill.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Assignment 1: Design a Long-Context Serving Architecture

Given a specific long-context use case, design a complete serving architecture.

In [ ]:
# ASSIGNMENT 1: Long-Context Architecture Design

class LongContextArchitect:
    """
    TODO: Design a serving architecture for long-context models.

    Consider:
    - How many GPUs needed for the model + KV-cache?
    - Ring attention vs tensor parallelism vs sequence parallelism?
    - Offloading strategy?
    - Chunked prefill configuration?
    """

    def __init__(self, model: LongContextModel, target_context: int,
                 target_ttft_s: float, gpu_memory_gb: float = 80):
        self.model = model
        self.target_context = target_context
        self.target_ttft = target_ttft_s
        self.gpu_memory = gpu_memory_gb

    def compute_requirements(self) -> dict:
        """
        TODO: Calculate total resource requirements.

        Return:
        - min_gpus_memory: Minimum GPUs for memory
        - min_gpus_latency: Minimum GPUs for latency target
        - recommended_config: dict with architecture details
        """
        # YOUR CODE HERE
        model_mem = self.model.model_size_gb
        kv_mem = self.model.kv_cache_gb(self.target_context)
        total_mem = model_mem + kv_mem

        # Memory requirement
        min_gpus_memory = int(np.ceil(total_mem / (self.gpu_memory * 0.9)))  # 90% utilization

        # Latency requirement
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        attn_flops = self.model.attention_flops(self.target_context)
        ffn_flops = 2 * total_params * self.target_context
        total_flops = attn_flops + ffn_flops
        single_gpu_time = total_flops / (989 * 1e12)  # H100
        min_gpus_latency = int(np.ceil(single_gpu_time / self.target_ttft))

        recommended_gpus = max(min_gpus_memory, min_gpus_latency)

        return {
            'model_memory_gb': model_mem,
            'kv_cache_memory_gb': kv_mem,
            'total_memory_gb': total_mem,
            'min_gpus_memory': min_gpus_memory,
            'min_gpus_latency': min_gpus_latency,
            'recommended_gpus': recommended_gpus,
            'strategy': 'ring_attention' if recommended_gpus >= 8 else 'tensor_parallel',
            'chunk_size': min(8192, self.target_context // 4),
            'offloading': 'cpu' if kv_mem > self.gpu_memory * recommended_gpus * 0.5 else 'none',
        }


# Test for different scenarios
scenarios = [
    ('8B model, 128K ctx, 5s TTFT', models['LLaMA-3.1-8B'], 131072, 5.0),
    ('70B model, 128K ctx, 10s TTFT', models['LLaMA-3.1-70B'], 131072, 10.0),
    ('70B model, 1M ctx, 30s TTFT', models['Gemini-1M'], 1048576, 30.0),
]

for desc, model, ctx, ttft in scenarios:
    architect = LongContextArchitect(model, ctx, ttft)
    reqs = architect.compute_requirements()
    print(f"\n{'='*60}")
    print(f" {desc}")
    print(f"{'='*60}")
    for k, v in reqs.items():
        if isinstance(v, float):
            print(f"  {k:30s}: {v:.1f}")
        else:
            print(f"  {k:30s}: {v}")

---
## Assignment 2: Benchmark Long-Context Performance

In [ ]:
# ASSIGNMENT 2: Long-Context Benchmarking Framework

class LongContextBenchmark:
    """
    TODO: Benchmark different strategies for long-context serving.

    Strategies to compare:
    1. Standard attention (single GPU, if possible)
    2. Ring attention (multi-GPU)
    3. Chunked prefill
    4. With/without offloading
    """

    def __init__(self, model: LongContextModel, gpu_config: dict):
        self.model = model
        self.gpu = gpu_config

    def benchmark_strategy(self, context_length: int, strategy: str,
                           num_gpus: int = 1) -> dict:
        """
        TODO: Estimate performance for a given strategy.

        Returns:
        - prefill_time: seconds
        - decode_tpot: time per output token
        - peak_memory_gb: per GPU
        - feasible: boolean
        """
        # YOUR CODE HERE
        total_params = 12 * self.model.num_layers * self.model.hidden_dim ** 2
        attn_flops = self.model.attention_flops(context_length)
        ffn_flops = 2 * total_params * context_length

        if strategy == 'standard':
            prefill_time = (attn_flops + ffn_flops) / (self.gpu['tflops'] * 1e12)
            peak_memory = self.model.model_size_gb + self.model.kv_cache_gb(context_length)
        elif strategy == 'ring_attention':
            # Distribute attention across GPUs
            chunk = context_length // num_gpus
            # Each GPU: chunk^2 attention per step, num_gpus steps
            per_step_flops = 2 * chunk * chunk * self.model.head_dim * self.model.num_kv_heads * self.model.num_layers
            per_step_ffn = 2 * total_params * chunk
            per_step_time = (per_step_flops + per_step_ffn / num_gpus) / (self.gpu['tflops'] * 1e12)
            prefill_time = per_step_time * num_gpus
            peak_memory = self.model.model_size_gb + self.model.kv_cache_gb(context_length) / num_gpus
        elif strategy == 'chunked':
            cp = ChunkedPrefill(self.model, compute_tflops=self.gpu['tflops'])
            result = cp.simulate_chunked_prefill(context_length, 8192)
            prefill_time = result['total_time']
            peak_memory = self.model.model_size_gb + self.model.kv_cache_gb(context_length)
        else:
            prefill_time = 0
            peak_memory = 0

        # Decode TPOT
        decode_bytes = total_params * self.model.param_bytes
        decode_tpot = decode_bytes / (self.gpu['bandwidth_gbps'] * 1e9 * num_gpus)

        feasible = peak_memory < self.gpu['memory_gb'] * num_gpus * 0.95

        return {
            'strategy': strategy,
            'num_gpus': num_gpus,
            'prefill_time_s': prefill_time,
            'decode_tpot_ms': decode_tpot * 1000,
            'peak_memory_gb': peak_memory,
            'feasible': feasible,
        }


bench = LongContextBenchmark(
    models['LLaMA-3.1-70B'],
    {'tflops': 989, 'bandwidth_gbps': 3350, 'memory_gb': 80}
)

test_contexts = [8192, 32768, 131072]
test_strategies = [
    ('standard', 1), ('standard', 8),
    ('ring_attention', 4), ('ring_attention', 8), ('ring_attention', 16),
    ('chunked', 1), ('chunked', 8),
]

print(f"{'Context':>10} {'Strategy':>15} {'GPUs':>5} {'Prefill(s)':>12} {'TPOT(ms)':>10} "
      f"{'Memory(GB)':>12} {'Feasible':>8}")
print("-" * 80)
for ctx in test_contexts:
    for strat, gpus in test_strategies:
        r = bench.benchmark_strategy(ctx, strat, gpus)
        print(f"{ctx:>10,} {strat:>15} {gpus:>5} {r['prefill_time_s']:>10.2f}s "
              f"{r['decode_tpot_ms']:>8.2f}ms {r['peak_memory_gb']:>10.1f}GB "
              f"{'YES' if r['feasible'] else 'NO':>8}")
    print()

---
## Assignment 3: Implement a KV-Cache Offloading Strategy

In [ ]:
# ASSIGNMENT 3: Smart KV-Cache Offloading

class SmartOffloader:
    """
    TODO: Implement a smarter offloading strategy that:
    1. Uses attention scores to decide what to keep on GPU
    2. Prefetches KV-cache blocks predicted to be needed
    3. Supports async transfers overlapped with computation
    """

    def __init__(self, gpu_budget_tokens: int, total_tokens: int,
                 prefetch_window: int = 4,
                 gpu_to_cpu_bw_gbps: float = 64.0,
                 kv_per_token_bytes: int = 327680):
        self.gpu_budget = gpu_budget_tokens
        self.total_tokens = total_tokens
        self.prefetch_window = prefetch_window
        self.bw = gpu_to_cpu_bw_gbps
        self.kv_per_token = kv_per_token_bytes

        # YOUR CODE HERE: Initialize data structures
        self.on_gpu = set()  # Token indices on GPU
        self.importance = np.zeros(total_tokens)  # Importance scores
        self.access_history = []
        self.prefetch_queue = []

        self.hits = 0
        self.misses = 0
        self.prefetch_hits = 0

    def update_importance(self, attention_scores: np.ndarray):
        """
        TODO: Update token importance based on attention scores.
        Use exponential moving average to track importance over time.
        """
        # YOUR CODE HERE
        alpha = 0.3
        self.importance = (1 - alpha) * self.importance
        for idx in range(min(len(attention_scores), self.total_tokens)):
            self.importance[idx] += alpha * attention_scores[idx]

    def decide_gpu_contents(self) -> set:
        """
        TODO: Decide which tokens should be on GPU.
        Keep: sink tokens + most important + recent + prefetch candidates
        """
        # YOUR CODE HERE
        # Always keep first 4 (sinks) and last recent_window
        keep = set(range(4))
        recent = set(range(max(0, self.total_tokens - self.gpu_budget // 2), self.total_tokens))
        keep.update(recent)

        # Fill remaining budget with highest importance
        remaining = self.gpu_budget - len(keep)
        if remaining > 0:
            candidates = set(range(self.total_tokens)) - keep
            top = sorted(candidates, key=lambda i: self.importance[i], reverse=True)[:remaining]
            keep.update(top)

        self.on_gpu = keep
        return keep

    def access(self, token_idx: int) -> dict:
        """
        TODO: Access a token's KV-cache.
        Track hits/misses, trigger prefetch for nearby tokens.
        """
        self.access_history.append(token_idx)

        if token_idx in self.on_gpu:
            self.hits += 1
            latency = 0
        elif token_idx in self.prefetch_queue:
            self.prefetch_hits += 1
            latency = self.kv_per_token / (self.bw * 1e9) * 0.5  # Partially prefetched
        else:
            self.misses += 1
            latency = self.kv_per_token / (self.bw * 1e9)

        # Trigger prefetch for nearby tokens
        self.prefetch_queue = list(range(
            max(0, token_idx - self.prefetch_window),
            min(self.total_tokens, token_idx + self.prefetch_window)
        ))

        return {'latency_ms': latency * 1000, 'hit': token_idx in self.on_gpu}


# Test smart offloader
smart = SmartOffloader(gpu_budget_tokens=16384, total_tokens=131072)

# Simulate attention patterns that reveal token importance
fake_attn = np.random.exponential(0.001, 131072)
fake_attn[:4] = 0.1  # Sinks
fake_attn[1000:1100] = 0.05  # Some important content
fake_attn[-2000:] = 0.02  # Recent context

smart.update_importance(fake_attn)
smart.decide_gpu_contents()

# Simulate decode accesses
total_latency = 0
for _ in range(1000):
    r = np.random.random()
    if r < 0.6:
        token = np.random.randint(131072 - 2000, 131072)
    elif r < 0.8:
        token = np.random.randint(1000, 1100)
    elif r < 0.9:
        token = np.random.randint(0, 4)
    else:
        token = np.random.randint(0, 131072)
    result = smart.access(token)
    total_latency += result['latency_ms']

total_accesses = smart.hits + smart.misses + smart.prefetch_hits
print(f"Smart Offloader Results:")
print(f"  GPU hits: {smart.hits}/{total_accesses} ({smart.hits/total_accesses:.1%})")
print(f"  Prefetch hits: {smart.prefetch_hits}/{total_accesses} ({smart.prefetch_hits/total_accesses:.1%})")
print(f"  Misses: {smart.misses}/{total_accesses} ({smart.misses/total_accesses:.1%})")
print(f"  Total transfer latency: {total_latency:.1f} ms")
print(f"  Avg latency per access: {total_latency/total_accesses:.3f} ms")

## Summary

### Key Takeaways

1. **Long-context serving** is fundamentally limited by O(n^2) attention complexity and linear KV-cache memory growth

2. **Ring Attention** effectively distributes both memory and compute across devices, achieving near-linear speedup for long sequences

3. **Chunked prefill** manages memory peaks but later chunks take longer due to attending to all prior context

4. **KV-cache offloading** enables sequences beyond GPU memory capacity, with smart placement based on access patterns significantly reducing transfer overhead

5. For **1M+ context**, you need both distributed compute (ring attention) and memory hierarchy management (offloading)

### Further Reading
- Ring Attention: https://arxiv.org/abs/2310.01889
- Sequence Parallelism: https://arxiv.org/abs/2105.13120
- InfiniGen: Efficient Generative Inference of Large Language Models with Dynamic KV Cache Management